In [ ]:
!pip install -q pandas numpy scikit-learn xgboost lightgbm \
 matplotlib seaborn tsfresh adversarial-robustness-toolbox

In [ ]:
from google.colab import drive
import os
import shutil

mountpoint = '/content/drive'

if os.path.exists(mountpoint):
    try:
        if os.path.ismount(mountpoint):
            drive.flush_and_unmount()
        elif os.path.isdir(mountpoint) and os.listdir(mountpoint):
            for item in os.listdir(mountpoint):
                item_path = os.path.join(mountpoint, item)
                if os.path.isfile(item_path) or os.path.islink(item_path):
                    os.remove(item_path)
                elif os.path.isdir(item_path):
                    shutil.rmtree(item_path)
    except Exception as e:
        print(f"Error: {e}")

os.makedirs(mountpoint, exist_ok=True)

drive.mount(mountpoint, force_remount=True)
PROJECT_DIR = os.path.join(mountpoint, 'MyDrive/man-bot-detection')

In [ ]:
!pip install -q datasets huggingface_hub

import os
from datasets import load_dataset, load_from_disk

CAPTCHA_CACHE_DIR = '/content/drive/MyDrive/man-bot-detection/data/raw/captchasolve30k_hf_cache'

if os.path.isdir(CAPTCHA_CACHE_DIR) and os.listdir(CAPTCHA_CACHE_DIR):
    dataset = load_from_disk(CAPTCHA_CACHE_DIR)
else:
    dataset = load_dataset("Capycap-AI/CaptchaSolve30k")
    os.makedirs(CAPTCHA_CACHE_DIR, exist_ok=True)
    dataset.save_to_disk(CAPTCHA_CACHE_DIR)

print(dataset)
print(f"Кількість зразків у тренувальному наборі: {len(dataset['train'])}")
print(dataset['train'][0])

In [ ]:
!wget https://raw.githubusercontent.com/chrisgdt/DELBOT-Mouse/master/python/parseSamples.py
!wget https://raw.githubusercontent.com/chrisgdt/DELBOT-Mouse/master/python/ganParser.py
!wget https://raw.githubusercontent.com/chrisgdt/DELBOT-Mouse/master/python/botDrawing.py

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

target_folder = '/content/drive/MyDrive/man-bot-detection/data/raw/delbot_mouse'

import os
os.makedirs(target_folder, exist_ok=True)

already_present = any(
    os.path.isdir(os.path.join(target_folder, d)) and d.startswith('circles_')
    for d in os.listdir(target_folder)
) if os.path.isdir(target_folder) else False

if not already_present:
    archive_path = os.path.join(target_folder, 'dataset.tar.gz')
    !wget -O "$archive_path" https://github.com/chrisgdt/DELBOT-Mouse/raw/master/python/dataset.tar.gz

    archive_size = os.path.getsize(archive_path) if os.path.exists(archive_path) else 0
    if archive_size < 100_000:
        raise RuntimeError(
            f"Завантажений файл підозріло малий ({archive_size} байт) — це, ймовірно, "
            f"HTML-сторінка помилки, а не dataset.tar.gz. Перевір посилання вручну: "
            f"https://github.com/chrisgdt/DELBOT-Mouse/tree/master/python"
        )

    !tar -xzvf "$archive_path" -C "$target_folder"

    os.remove(archive_path)

!ls -la "$target_folder"

In [ ]:
!ls -la "$target_folder"

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

target_folder = '/content/drive/MyDrive/man-bot-detection/data/raw/cic_ids_2017'

import os
os.makedirs(target_folder, exist_ok=True)

zip_path = os.path.join(target_folder, 'MachineLearningCSV.zip')

already_present = any(f.endswith('.csv') for f in os.listdir(target_folder)) if os.path.isdir(target_folder) else False

if not already_present:
    download_urls = [
        "https://cicresearch.ca/CICDataset/CIC-IDS-2017/Dataset/CIC-IDS-2017/CSVs/MachineLearningCSV.zip",
        "http://205.174.165.80/CICDataset/CIC-IDS-2017/Dataset/CIC-IDS-2017/CSVs/MachineLearningCSV.zip",
    ]

    MIN_VALID_SIZE = 100_000_000

    downloaded_ok = False
    for url in download_urls:
        !wget -O "$zip_path" "$url"
        size = os.path.getsize(zip_path) if os.path.exists(zip_path) else 0
        print(f"Розмір: {size / 1024 / 1024:.1f} МБ")
        if size >= MIN_VALID_SIZE:
            downloaded_ok = True
            break

    if not downloaded_ok:
        raise RuntimeError(
            "Не вдалося завантажити справжній архів MachineLearningCSV.zip з жодного джерела "
            "(отримано HTML-сторінку замість файлу з обох спроб). "
            "Завантаж архів вручну зі сторінки "
            "https://www.unb.ca/cic/datasets/ids-2017.html і поклади його за шляхом:\n"
            f"{zip_path}"
        )

    !unzip -o "$zip_path" -d "$target_folder"
    os.remove(zip_path)

!ls -la "$target_folder"

In [ ]:
import os
import pandas as pd
import numpy as np
import io
from google.colab import drive

ROOT_PATH = "/content/drive/MyDrive/man-bot-detection/data/raw/"

MAX_SIZE_MB = 500
SAMPLE_ROWS = 1000

def detect_format(filepath):
    ext = os.path.splitext(filepath)[1].lower()
    if ext == '.csv':
        return 'csv'
    elif ext == '.parquet':
        return 'parquet'
    elif ext in ['.log', '.txt', '.labeled']:
        with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
            first_line = f.readline().strip()
            if first_line.startswith('#') or '#fields' in first_line:
                return 'zeek'
            else:
                return 'tsv'
    else:
        return None

def read_zeek_log(filepath, nrows=None):
    columns = None
    with open(filepath, 'r', encoding='utf-8', errors='ignore') as f:
        for line in f:
            if line.startswith('#fields'):
                columns = line.strip().split('\t')[1:]
                break
            elif not line.startswith('#'):
                columns = [
                    'ts', 'uid', 'id.orig_h', 'id.orig_p', 'id.resp_h', 'id.resp_p',
                    'proto', 'service', 'duration', 'orig_bytes', 'resp_bytes', 'conn_state',
                    'local_orig', 'local_resp', 'missed_bytes', 'history', 'orig_pkts',
                    'orig_ip_bytes', 'resp_pkts', 'resp_ip_bytes', 'tunnel_parents',
                    'label', 'detailed-label'
                ]
                break

    if columns is None:
        raise ValueError("Не вдалося визначити колонки для Zeek-логу")

    df = pd.read_csv(
        filepath,
        sep='\t',
        header=None,
        names=columns,
        comment='#',
        nrows=nrows,
        low_memory=False,
        dtype=str
    )
    return df

def read_file_smart(filepath):
    size_mb = os.path.getsize(filepath) / (1024**2)
    is_large = size_mb > MAX_SIZE_MB

    fmt = detect_format(filepath)
    if fmt is None:
        return None, f"Формат не підтримується ({os.path.basename(filepath)})"

    try:
        if fmt == 'csv':
            df = pd.read_csv(filepath, nrows=None if not is_large else SAMPLE_ROWS, low_memory=False)
        elif fmt == 'parquet':
            df = pd.read_parquet(filepath, engine='pyarrow')
        elif fmt == 'zeek':
            df = read_zeek_log(filepath, nrows=None if not is_large else SAMPLE_ROWS)
        else:
            df = pd.read_csv(filepath, sep='\t', nrows=None if not is_large else SAMPLE_ROWS, low_memory=False)

        is_sample = is_large and len(df) >= SAMPLE_ROWS
        msg = f"{'ВЕСЬ файл' if not is_sample else 'ТІЛЬКИ ПЕРШІ РЯДКИ (sample)'} | Розмір: {size_mb:.1f} MB | Рядків у завантаженій частині: {len(df)}"
        return df, msg

    except Exception as e:
        return None, f"ПОМИЛКА: {str(e)[:200]}..."

results = []

for root, dirs, files in os.walk(ROOT_PATH):
    for file in files:
        if file.startswith('.') or file in ['README.md', 'README.txt', 'LICENSE']:
            continue

        if any(file.endswith(ext) for ext in ['.zip', '.pcap', '.pcapng', '.tar', '.gz']):
            continue

        full_path = os.path.join(root, file)
        rel_path = os.path.relpath(full_path, ROOT_PATH)

        print("\n" + "="*80)
        print(f"ФАЙЛ: {rel_path}")
        print("="*80)

        df, msg = read_file_smart(full_path)

        if df is None or df.empty:
            print(msg)
            continue

        print(f"\n{msg}")
        print(f"Формат: {detect_format(full_path)}")
        print(f"Колонок: {len(df.columns)}")
        print(f"\nSHAPE: {df.shape}")

        print(df.head())

        buf = io.StringIO()
        df.info(verbose=True, show_counts=True, buf=buf)
        info_text = buf.getvalue()
        lines = info_text.split('\n')
        print('\n'.join(lines[-10:]))

        missing = df.isnull().sum()
        missing_pct = (missing / len(df)) * 100
        missing_report = missing[missing > 0]
        if not missing_report.empty:
            for col, cnt in missing_report.items():
                print(f"   {col}: {cnt} ({missing_pct[col]:.1f}%)")

        mem_usage = df.memory_usage(deep=True).sum() / (1024**2)
        print(f"\nЗайнято пам'яті: {mem_usage:.1f} MB")

        results.append({
            'file': rel_path,
            'rows': len(df),
            'cols': len(df.columns),
            'size_mb': os.path.getsize(full_path) / (1024**2),
            'missing_cols': len(missing_report),
            'memory_mb': mem_usage
        })

print("\n" + "="*80)
print("ПІДСУМКОВИЙ ЗВІТ ЗА ВСІМА ФАЙЛАМИ")
print("="*80)

df_summary = pd.DataFrame(results)
if not df_summary.empty:
    print(df_summary.to_string(index=False))
else:
    print("Не знайдено жодного файлу для аналізу.")

In [ ]:
import os
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

delbot_raw = '/content/drive/MyDrive/man-bot-detection/data/raw/delbot_mouse/'

bot_count = 0
human_count_delbot = 0

for dir_name in os.listdir(delbot_raw):
    dir_path = os.path.join(delbot_raw, dir_name)
    if not os.path.isdir(dir_path):
        continue

    txt_files = [f for f in os.listdir(dir_path) if f.endswith('.txt')]
    count = len(txt_files)

    if 'bot' in dir_name.lower():
        bot_count += count
    elif 'human' in dir_name.lower():
        human_count_delbot += count

print(f"Бот-сесій (DELBOT-Mouse): {bot_count}")
print(f"Людських сесій (DELBOT-Mouse): {human_count_delbot}")

captcha_human = 5000
total_human = captcha_human + human_count_delbot
total_bot = bot_count

print(f"Люди (Captcha + DELBOT-human): {total_human}")
print(f"Боти (DELBOT-bot): {total_bot}")

data = {'label': ['human'] * total_human + ['bot'] * total_bot}
df_counts = pd.DataFrame(data)

plt.figure(figsize=(6, 4))
sns.countplot(data=df_counts, x='label', palette=['#3498db', '#e74c3c'])
plt.title('Кількість сесій: люди vs боти', fontsize=14)
plt.xlabel('Клас')
plt.ylabel('Кількість сесій')
plt.grid(axis='y', linestyle='--', alpha=0.7)

for p in plt.gca().patches:
    plt.gca().annotate(f'{int(p.get_height())}',
                       (p.get_x() + p.get_width()/2., p.get_height()),
                       ha='center', va='bottom', fontsize=12)

plt.tight_layout()

fig_dir = '/content/drive/MyDrive/man-bot-detection/results/figures'
os.makedirs(fig_dir, exist_ok=True)
plt.savefig(os.path.join(fig_dir, 'class_balance.png'), dpi=150)
plt.show()

In [ ]:
import os
import matplotlib.pyplot as plt

delbot_raw = '/content/drive/MyDrive/man-bot-detection/data/raw/delbot_mouse/'

def read_delbot_trajectory(file_path, max_points=500):
    xs, ys = [], []
    with open(file_path, 'r') as f:
        lines = f.readlines()
        for line in lines[1:]:
            parts = line.strip().split(',')
            if len(parts) >= 4:
                try:
                    event_type = parts[1].strip()
                    if event_type == 'Move':
                        x = float(parts[2])
                        y = float(parts[3])
                        xs.append(x)
                        ys.append(y)
                        if len(xs) >= max_points:
                            break
                except ValueError:
                    continue
    return xs, ys

def get_first_file(dir_path, ext='.txt'):
    for f in os.listdir(dir_path):
        if f.endswith(ext):
            return os.path.join(dir_path, f)
    return None

human_trajectories = []
bot_trajectories = []

human_dirs = [d for d in os.listdir(delbot_raw)
              if os.path.isdir(os.path.join(delbot_raw, d)) and 'human' in d]

for d in human_dirs[:2]:
    dir_path = os.path.join(delbot_raw, d)
    file_path = get_first_file(dir_path)
    if file_path:
        xs, ys = read_delbot_trajectory(file_path)
        if xs and ys:
            human_trajectories.append((xs, ys))

bot_dirs = [d for d in os.listdir(delbot_raw)
            if os.path.isdir(os.path.join(delbot_raw, d)) and 'bot' in d]

for d in bot_dirs[:2]:
    dir_path = os.path.join(delbot_raw, d)
    file_path = get_first_file(dir_path)
    if file_path:
        xs, ys = read_delbot_trajectory(file_path)
        if xs and ys:
            bot_trajectories.append((xs, ys))

print(f"Завантажено {len(human_trajectories)} людських траєкторій")
print(f"Завантажено {len(bot_trajectories)} бот-траєкторій")

plt.figure(figsize=(12, 5))

plt.subplot(1, 2, 1)
for xs, ys in human_trajectories:
    plt.plot(xs, ys, alpha=0.7, linewidth=1)
plt.title('Людські траєкторії (DELBOT-Mouse)', fontsize=12)
plt.xlabel('X координата')
plt.ylabel('Y координата')
plt.axis('equal')
plt.grid(True, alpha=0.3)

plt.subplot(1, 2, 2)
for xs, ys in bot_trajectories:
    plt.plot(xs, ys, alpha=0.7, linewidth=1)
plt.title('Бот-траєкторії (DELBOT-Mouse)', fontsize=12)
plt.xlabel('X координата')
plt.ylabel('Y координата')
plt.axis('equal')
plt.grid(True, alpha=0.3)

plt.tight_layout()

save_path = '/content/drive/MyDrive/man-bot-detection/results/figures/trajectories_example.png'
os.makedirs(os.path.dirname(save_path), exist_ok=True)
plt.savefig(save_path, dpi=150)
plt.show()

In [ ]:
from google.colab import drive
drive.mount('/content/drive', force_remount=True)

import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns
import os
import glob

sns.set_style("darkgrid")
plt.rcParams['figure.figsize'] = (14, 8)

base_path = '/content/drive/MyDrive/man-bot-detection/data/raw'

def load_csvs_from_folder(folder_path):
    all_files = glob.glob(os.path.join(folder_path, '**', '*.csv'), recursive=True)
    if not all_files:
        return None
    df_list = []
    for file in all_files:
        try:
            df = pd.read_csv(file, encoding='utf-8')
        except UnicodeDecodeError:
            df = pd.read_csv(file, encoding='latin-1')
        df_list.append(df)
    combined = pd.concat(df_list, ignore_index=True)
    return combined

cic_path = os.path.join(base_path, 'cic_ids_2017')
cic_df = load_csvs_from_folder(cic_path)

unsw_path = os.path.join(base_path, 'unsw_nb15')
unsw_df = load_csvs_from_folder(unsw_path)

if cic_df is None or unsw_df is None:
    raise SystemExit("Не вдалося завантажити один із датасетів. Перевірте шлях.")

cic_label_col = None
for col in cic_df.columns:
    if col.strip().lower() == 'label':
        cic_label_col = col
        break
if cic_label_col is None:
    raise ValueError("У CIC-IDS2017 не знайдено колонку з мітками (Label).")
unsw_label_col = None
for col in unsw_df.columns:
    if col.strip().lower() == 'attack_cat' or col.strip().lower() == 'label':
        unsw_label_col = col
        break
if unsw_label_col is None:
    raise ValueError("У UNSW-NB15 не знайдено колонку з мітками (attack_cat або label).")

cic_counts = cic_df[cic_label_col].value_counts().reset_index()
cic_counts.columns = ['Attack_Type', 'Count']
print(cic_counts)

unsw_counts = unsw_df[unsw_label_col].value_counts().reset_index()
unsw_counts.columns = ['Attack_Type', 'Count']
print(unsw_counts)

fig, axes = plt.subplots(1, 2, figsize=(18, 9))

sns.barplot(data=cic_counts.sort_values('Count', ascending=False), y='Attack_Type', x='Count', ax=axes[0], palette='viridis')
axes[0].set_title('CIC-IDS2017 – кількість зразків за типами атак', fontsize=16)
axes[0].set_xlabel('Кількість', fontsize=12)
axes[0].set_ylabel('Тип атаки', fontsize=12)
for index, row in cic_counts.sort_values('Count', ascending=False).iterrows():
    axes[0].text(row.Count, index, f' {row.Count}', color='black', ha="left", va="center")

sns.barplot(data=unsw_counts.sort_values('Count', ascending=False), y='Attack_Type', x='Count', ax=axes[1], palette='plasma')
axes[1].set_title('UNSW-NB15 – кількість зразків за типами атак', fontsize=16)
axes[1].set_xlabel('Кількість', fontsize=12)
axes[1].set_ylabel('Тип атаки', fontsize=12)
for index, row in unsw_counts.sort_values('Count', ascending=False).iterrows():
    axes[1].text(row.Count, index, f' {row.Count}', color='black', ha="left", va="center")

plt.tight_layout()
plt.show()

top_cic = cic_counts.sort_values('Count', ascending=False).head(9)
others_count_cic = cic_counts['Count'].sum() - top_cic['Count'].sum()
if others_count_cic > 0:
    top_cic = pd.concat([top_cic, pd.DataFrame([['Інші', others_count_cic]], columns=['Attack_Type', 'Count'])], ignore_index=True)

explode_cic = [0.05 if val < top_cic['Count'].max() * 0.1 else 0 for val in top_cic['Count']]

plt.figure(figsize=(12, 12))
plt.pie(top_cic['Count'], labels=top_cic['Attack_Type'], autopct='%1.1f%%', startangle=90, explode=explode_cic, pctdistance=0.85, colors=sns.color_palette('tab10'))
plt.title('CIC-IDS2017 – розподіл атак (топ-9 + інші)', fontsize=18)
plt.axis('equal')
plt.show()

top_unsw = unsw_counts.sort_values('Count', ascending=False).head(9)
others_unsw = unsw_counts['Count'].sum() - top_unsw['Count'].sum()
if others_unsw > 0:
    top_unsw = pd.concat([top_unsw, pd.DataFrame([['Інші', others_unsw]], columns=['Attack_Type', 'Count'])], ignore_index=True)

explode_unsw = [0.05 if val < top_unsw['Count'].max() * 0.1 else 0 for val in top_unsw['Count']]

plt.figure(figsize=(12, 12))
plt.pie(top_unsw['Count'], labels=top_unsw['Attack_Type'], autopct='%1.1f%%', startangle=90, explode=explode_unsw, pctdistance=0.85, colors=sns.color_palette('pastel'))
plt.title('UNSW-NB15 – розподіл атак (топ-9 + інші)', fontsize=18)
plt.axis('equal')
plt.show()